# 01 — Data Preprocessing

**Tujuan:** Load raw dataset LinkedIn dari Kaggle, gabungkan multi-file, filter domain IT, handle missing values, dan simpan sebagai dataset bersih untuk tahap berikutnya.

**Dataset:**
1. [arshkon/linkedin-job-postings](https://www.kaggle.com/datasets/arshkon/linkedin-job-postings) — 124,000+ LinkedIn job postings (2023-2024)
2. [asaniczka/data-science-job-postings-and-skills](https://www.kaggle.com/datasets/asaniczka/data-science-job-postings-and-skills) — Data Science focused

**Output:** `data/processed/jobs_clean.csv`

In [ ]:
try:
    df = load_arshkon_dataset()
    print(f'Loaded REAL LinkedIn data: {len(df):,} rows')
except FileNotFoundError:
    print('⚠️  Real Kaggle dataset not found in data/raw/')
    print('Falling back to synthetic demo dataset for notebook walkthrough.')
    print('To use real data, download from Kaggle and place in data/raw/.')
    from src.data_loader import _generate_synthetic_jobs
    df = _generate_synthetic_jobs(n=2000)
    print(f'Loaded SYNTHETIC data: {len(df):,} rows')

print(f'\nColumns: {df.columns.tolist()}')
df.head(3)

## 1.1 Load Raw Data

In [ ]:
df = load_arshkon_dataset()
print(f'Total rows: {len(df):,}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

## 1.2 Missing Value Analysis

In [ ]:
missing = df.isnull().sum() / len(df) * 100
missing = missing[missing > 0].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
missing.plot(kind='barh', ax=ax, color='#0A66C2')
ax.set_xlabel('Missing %')
ax.set_title('Missing Values per Column')
plt.tight_layout()
plt.show()
print(missing.head(10))

## 1.3 Filter IT Domain Jobs

In [ ]:
df_it = filter_it_jobs(df, title_col='title')
print(f'Before filter: {len(df):,} jobs')
print(f'After filter (IT only): {len(df_it):,} jobs')
print(f'Retention rate: {len(df_it)/len(df)*100:.1f}%')

# Top job titles
df_it['title'].value_counts().head(15)

## 1.4 Clean Skills Column

In [ ]:
if 'skills_abr' in df_it.columns:
    df_it = clean_skills_column(df_it, skills_col='skills_abr')
    print('Sample skills_list:')
    print(df_it[['title', 'skills_list']].head())

## 1.5 Handle Missing Salary

In [ ]:
df_it = handle_missing_salary(df_it)

if 'med_salary' in df_it.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    df_it['med_salary'].dropna().plot(kind='hist', bins=50, ax=ax, color='#0A66C2')
    ax.set_xlabel('Salary (USD/year)')
    ax.set_title('Distribution of Median Salary')
    plt.tight_layout()
    plt.show()

## 1.6 Normalize Categorical Fields

In [ ]:
df_it = normalize_experience_level(df_it)
df_it = normalize_work_type(df_it)

if 'experience_level' in df_it.columns:
    print('Experience level distribution:')
    print(df_it['experience_level'].value_counts())

## 1.7 Save Processed Data

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
output_path = PROCESSED_DIR / 'jobs_clean.csv'
df_it.to_csv(output_path, index=False)
print(f'Saved {len(df_it):,} rows to {output_path}')
print(f'Final columns: {df_it.columns.tolist()}')

## Kesimpulan Preprocessing

- Mulai dari 124,000+ raw postings → setelah filter IT tersisa job yang relevan untuk sistem rekomendasi domain IT
- Skill column dikonversi dari string ke list of skills untuk memudahkan tahap feature engineering
- Missing salary diisi dengan median per title untuk mempertahankan jumlah job tanpa membuang data
- Output siap dipakai untuk notebook 02 (Feature Engineering)